In [1]:

!pip install -q scikit-survival

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sksurv.ensemble import GradientBoostingSurvivalAnalysis
from sksurv.util import Surv

import warnings
warnings.filterwarnings("ignore")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 66.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.1/222.1 kB 12.5 MB/s eta 0:00:00


In [35]:
import numpy as np
import pandas as pd
import warnings
import os
import time as timer
from sklearn.model_selection import StratifiedKFold
from sklearn.isotonic import IsotonicRegression
import lightgbm as lgb
from sksurv.ensemble import GradientBoostingSurvivalAnalysis
from sksurv.util import Surv
from sksurv.metrics import concordance_index_censored


In [2]:
train = pd.read_csv("train.csv")
test  = pd.read_csv("test.csv")
sub   = pd.read_csv("sample_submission.csv")

print(f"Train : {train.shape}")
print(f"Test  : {test.shape}")
print(f"Sub   : {sub.shape}")
print(f"\nHit rate: {train['event'].mean():.1%}  ({train['event'].sum()} hits / {len(train)} fires)")
print(f"\nTrain columns:\n{list(train.columns)}")

Train : (221, 37)
Test  : (95, 35)
Sub   : (95, 5)

Hit rate: 31.2%  (69 hits / 221 fires)

Train columns:
['event_id', 'num_perimeters_0_5h', 'dt_first_last_0_5h', 'low_temporal_resolution_0_5h', 'area_first_ha', 'area_growth_abs_0_5h', 'area_growth_rel_0_5h', 'area_growth_rate_ha_per_h', 'log1p_area_first', 'log1p_growth', 'log_area_ratio_0_5h', 'relative_growth_0_5h', 'radial_growth_m', 'radial_growth_rate_m_per_h', 'centroid_displacement_m', 'centroid_speed_m_per_h', 'spread_bearing_deg', 'spread_bearing_sin', 'spread_bearing_cos', 'dist_min_ci_0_5h', 'dist_std_ci_0_5h', 'dist_change_ci_0_5h', 'dist_slope_ci_0_5h', 'closing_speed_m_per_h', 'closing_speed_abs_m_per_h', 'projected_advance_m', 'dist_accel_m_per_h2', 'dist_fit_r2_0_5h', 'alignment_cos', 'alignment_abs', 'cross_track_component', 'along_track_speed', 'event_start_hour', 'event_start_dayofweek', 'event_start_month', 'time_to_hit_hours', 'event']


In [3]:
# =============================
# 3. Feature Engineering (YOUR FEATURES)
# =============================
DROP_COLS = [
    'event_id', 'time_to_hit_hours', 'event',
    'relative_growth_0_5h',
    'projected_advance_m',
    'dist_slope_ci_0_5h',
    'centroid_displacement_m',
    'centroid_speed_m_per_h',
    'area_growth_abs_0_5h',
    'dist_change_ci_0_5h',
    'closing_speed_abs_m_per_h',
]

def make_features(df):
    X = df.drop(columns=[c for c in DROP_COLS if c in df.columns], errors='ignore').copy()

    X['time_to_zone_est'] = df['dist_min_ci_0_5h'] / df['closing_speed_m_per_h'].clip(lower=0.5)
    X['growth_x_align']   = df['area_growth_rate_ha_per_h'] * df['alignment_abs']
    X['log_dist']         = np.log1p(df['dist_min_ci_0_5h'])
    X['is_closing']       = (df['closing_speed_m_per_h'] > 0).astype(int)
    X['close_and_aligned']= X['is_closing'] * df['alignment_abs']

    X['dist_per_growth']  = df['dist_min_ci_0_5h'] / df['area_growth_rate_ha_per_h'].clip(lower=0.1)
    X['speed_x_align']    = df['closing_speed_m_per_h'] * df['alignment_abs']
    X['risk_ratio']       = df['closing_speed_m_per_h'] / (df['dist_min_ci_0_5h'] + 1)
    X['growth_vs_distance']= df['area_growth_rate_ha_per_h'] / (df['dist_min_ci_0_5h'] + 1)

    X = X.replace([np.inf, -np.inf], np.nan).fillna(0)

    return X

X_train_df = make_features(train)
X_test_df  = make_features(test)

FEATURES = X_train_df.columns.tolist()

X_train = X_train_df[FEATURES]
X_test  = X_test_df[FEATURES]


In [29]:

# Raw Features for GBSA

X_surv_train = train.drop(columns=['event_id', 'event', 'time_to_hit_hours'])
X_surv_test  = test.drop(columns=['event_id'])

X_surv_train = X_surv_train.replace([np.inf, -np.inf], np.nan).fillna(0)
X_surv_test  = X_surv_test.replace([np.inf, -np.inf], np.nan).fillna(0)

y_surv = Surv.from_arrays(
    event=train['event'].astype(bool),
    time=train['time_to_hit_hours']
)

print(f"GBSA feature count: {X_surv_train.shape[1]}")


GBSA feature count: 34


In [30]:
HORIZONS_PRED= np.array([12, 24, 48, 72])



In [31]:

def get_surv_predictions(model, X):
    surv_fns = model.predict_survival_function(X)
    preds = np.zeros((len(surv_fns), len(HORIZONS_PRED)))

    for i, fn in enumerate(surv_fns):
        t_min, t_max = fn.domain
        for j, t in enumerate(HORIZONS_PRED):
            if t <= t_min:
                preds[i, j] = 1.0 - fn(t_min)
            elif t >= t_max:
                # Each fire gets ITS OWN survival value at t_max

                preds[i, j] = 1.0 - fn(t_max)
            else:
                preds[i, j] = 1.0 - fn(t)
    return preds

In [32]:
## trying to enforce monotonicity
def enforce_monotonicity(preds):
  preds = np.clip(preds, 0, 1)
  for i in range(1, preds.shape[1]):
    preds[:, i] = np.maximum(preds[:, i], preds[:, i-1])
    return preds

In [33]:
def weighted_brier_score(times, events, preds,
                          horizons=[24, 48, 72],
                          weights=[0.3, 0.4, 0.3]):
    horizon_to_col = {12: 0, 24: 1, 48: 2, 72: 3}
    total_bs = 0.0
    for h, w in zip(horizons, weights):
        col = horizon_to_col[h]
        bs_list = []
        for i in range(len(times)):
            if events[i] == 1 and times[i] <= h:
                bs_list.append((preds[i, col] - 1.0) ** 2)
            elif times[i] >= h:
                bs_list.append((preds[i, col] - 0.0) ** 2)
        total_bs += w * np.mean(bs_list)
    return total_bs

def competition_score(times, events, preds):
    risk_scores = (0.4 * preds[:, 0] +
                   0.3 * preds[:, 1] +
                   0.2 * preds[:, 2] +
                   0.1 * preds[:, 3])
    c_idx = concordance_index_censored(
        events.astype(bool), times, risk_scores
    )[0]
    wbs   = weighted_brier_score(times, events, preds)
    hybrid = 0.3 * c_idx + 0.7 * (1.0 - wbs)
    print(f"  C-index        : {c_idx:.4f}")
    print(f"  Weighted Brier : {wbs:.4f}")
    print(f"  Hybrid Score   : {hybrid:.4f}")
    return hybrid


In [47]:
# 5. GBSA Ensemble
# 3 configs x 6 seeds x 5 folds = 90 models

gbsa_configs = [
    {'learning_rate': 0.01,  'max_depth': 3, 'n_estimators': 1200,
     'subsample': 0.70, 'min_samples_leaf': 12, 'min_samples_split': 3},
    {'learning_rate': 0.01,  'max_depth': 3, 'n_estimators': 1200,
     'subsample': 0.85, 'min_samples_leaf': 15, 'min_samples_split': 3},
    {'learning_rate': 0.005, 'max_depth': 3, 'n_estimators': 1500,
     'subsample': 0.85, 'min_samples_leaf': 12, 'min_samples_split': 3},
]

SEEDS = [42, 123, 456, 777, 2024, 2025]

# Define event_values and time_values for stratification and scoring
event_values = train['event'].values
time_values = train['time_to_hit_hours'].values

# Accumulators
test_acc   = np.zeros((len(X_surv_test), 4))
oof_acc    = np.zeros((len(X_surv_train), 4))
oof_counts = np.zeros(len(X_surv_train))

total_runs = 0
total_jobs = len(gbsa_configs) * len(SEEDS) * 5
print(f"\nTotal models to train: {total_jobs}")
print("Starting training...\n")

t_start = timer.time()

for cfg_idx, cfg in enumerate(gbsa_configs):
    for seed in SEEDS:
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
        for fold_idx, (tr_idx, va_idx) in enumerate(
                cv.split(X_surv_train, event_values)):

            model = GradientBoostingSurvivalAnalysis(
                **cfg, random_state=seed
            )
            model.fit(
                X_surv_train.iloc[tr_idx],
                y_surv[tr_idx]
            )

            # OOF
            oof_acc[va_idx] += get_surv_predictions(
                model, X_surv_train.iloc[va_idx]
            )
            oof_counts[va_idx] += 1



            # Test
            test_acc += get_surv_predictions(model, X_surv_test)
            total_runs += 1

            if total_runs % 15 == 0 or total_runs == total_jobs:
                elapsed = timer.time() - t_start
                eta     = (elapsed / total_runs) * (total_jobs - total_runs)
                print(f"  [{total_runs:>3}/{total_jobs}] "
                      f"{elapsed/60:.1f}m elapsed  ~{eta/60:.1f}m left")

# Finalise
oof_final  = np.clip(oof_acc / oof_counts[:, None], 0, 1)
test_final = test_acc / total_runs

print(f"\n✓ Training done — {total_runs} models in {(timer.time()-t_start)/60:.1f}m")

# =============================
# 6. Calibration (his approach)
# =============================
# Power calibration on 24h only — applied to BOTH oof and test
test_final[:, 1] = np.clip(test_final[:, 1] ** 1.1, 0, 1)
oof_final[:, 1]  = np.clip(oof_final[:, 1]  ** 1.1, 0, 1)

# Monotonicity
test_final = enforce_monotonicity(test_final)
oof_final  = enforce_monotonicity(oof_final)

print("\nGBSA OOF Performance:")
competition_score(time_values, event_values, oof_final)

# =============================
# 7. Save GBSA submission
# =============================
sub_gbsa = pd.DataFrame({
    'event_id': test['event_id'],
    'prob_12h': test_final[:, 0],
    'prob_24h': test_final[:, 1],
    'prob_48h': test_final[:, 2],
    'prob_72h': test_final[:, 3],
})
sub_gbsa.to_csv("submission_gbsa_02.csv", index=False)
print("\n✓ submission_gbsa.csv saved")
print(sub_gbsa.describe().round(3).to_string())




Total models to train: 90
Starting training...

  [ 15/90] 0.4m elapsed  ~2.0m left
  [ 30/90] 0.8m elapsed  ~1.6m left
  [ 45/90] 1.4m elapsed  ~1.4m left
  [ 60/90] 2.0m elapsed  ~1.0m left
  [ 75/90] 2.5m elapsed  ~0.5m left
  [ 90/90] 3.0m elapsed  ~0.0m left

✓ Training done — 90 models in 3.0m

GBSA OOF Performance:
  C-index        : 0.9432
  Weighted Brier : 0.0150
  Hybrid Score   : 0.9725

✓ submission_gbsa.csv saved
           event_id  prob_12h  prob_24h  prob_48h  prob_72h
count  9.500000e+01    95.000    95.000    95.000    95.000
mean   5.695393e+07     0.208     0.278     0.311     0.366
std    2.329721e+07     0.313     0.387     0.398     0.400
min    1.066260e+07     0.016     0.027     0.047     0.091
25%    4.027536e+07     0.018     0.031     0.054     0.104
50%    5.480111e+07     0.020     0.034     0.058     0.111
75%    7.536942e+07     0.443     0.726     0.826     0.957
max    9.964946e+07     0.998     1.000     1.000     1.000
